In [1]:
import pandas as pd
import numpy as np

path = "data/raw/neighbourhood-profiles-2021-158-model.xlsx"
print(pd.ExcelFile(path).sheet_names)
raw = pd.read_excel(path, sheet_name="hd2021_census_profile", header=None)
print(raw.shape)
raw.iloc[:6, :4]

['hd2021_census_profile', 'Nbhdmetadata']
(2604, 159)


,0,1,2,3
0,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights
1,Neighbourhood Number,1,2,3
2,TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area
3,Total - Age groups of the population - 25% sam...,33300,31345,9850
4,0 to 14 years,4295,5690,1495
5,0 to 4 years,1460,1650,505


In [2]:
raw.iloc[[0, 1, 19, 32], :3]

,0,1,2
0,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown
1,Neighbourhood Number,1,2
19,65 years and over,5360,4170
32,65 years and over,16.1,13.3


In [3]:
labels = raw[0].astype(str).str.strip()
print("rows labelled '65 years and over':", (labels == "65 years and over").sum())

rows labelled '65 years and over': 6


In [4]:
def find_row(section, item):
    """Row number of `item`, searching only below the section heading `section`."""
    start = labels[labels == section].index
    assert len(start) == 1, f"heading not found exactly once: {section}"
    below = labels.loc[start[0] + 1:]
    return below[below == item].index[0]

def values(row):
    return pd.to_numeric(raw.loc[row, 1:], errors="coerce").to_numpy(dtype=float)

def section_total(section):
    return values(labels[labels == section].index[0])

In [5]:
AGE = "Total - Distribution (%) of the population by broad age groups - 25% sample data"
INC = "Total - Income statistics for private households - 25% sample data"
TEN = "Total - Private households by tenure - 25% sample data"
STR = "Total - Occupied private dwellings by structural type of dwelling - 25% sample data"
PER = "Total - Occupied private dwellings by period of construction - 25% sample data"

ctx = pd.DataFrame({
    "neighbourhood_id": raw.loc[1, 1:].astype(float).astype(int).astype(str).to_numpy(),
    "neighbourhood_name": raw.loc[0, 1:].to_numpy(),
    "median_income": values(find_row(INC, "Median total income of household in 2020 ($)")),
    "pct_65_plus": values(find_row(AGE, "65 years and over")),
    "pct_renters": values(find_row(TEN, "Renter")) / section_total(TEN) * 100,
    "pct_apt_5plus": values(find_row(STR, "Apartment in a building that has five or more storeys")) / section_total(STR) * 100,
    "pct_built_before_1980": (values(find_row(PER, "1960 or before")) + values(find_row(PER, "1961 to 1980"))) / section_total(PER) * 100,
})
ctx.head(3).round(1)

,neighbourhood_id,neighbourhood_name,median_income,pct_65_plus,pct_renters,pct_apt_5plus,pct_built_before_1980
0,1,West Humber-Clairville,92000.0,16.1,34.6,23.5,45.7
1,2,Mount Olive-Silverstone-Jamestown,76500.0,13.3,54.0,62.0,58.2
2,3,Thistletown-Beaumond Heights,86000.0,17.7,37.6,20.3,76.9


In [6]:
print(ctx.shape, "| missing values:", int(ctx.isna().sum().sum()))
bound = pd.read_csv("data/raw/Neighbourhoods - 4326.csv")
print("census numbers match boundary codes:", set(bound["AREA_SHORT_CODE"].astype(str)) == set(ctx["neighbourhood_id"]))
ctx.describe().round(1).T[["mean", "min", "max"]]

(158, 7) | missing values: 0
census numbers match boundary codes: True


,mean,min,max
median_income,88969.0,57200.0,222000.0
pct_65_plus,16.7,3.1,28.7
pct_renters,46.0,7.9,88.9
pct_apt_5plus,40.7,0.0,99.5
pct_built_before_1980,62.7,2.0,93.4


In [7]:
r = find_row(TEN, "Renter")
print(raw.loc[r, 1], "renters out of", raw.loc[labels[labels == TEN].index[0], 1], "households =",
      round(raw.loc[r, 1] / raw.loc[labels[labels == TEN].index[0], 1] * 100, 1), "%")

3705 renters out of 10700 households = 34.6 %


In [8]:
ctx.to_csv("data/processed/neighbourhood_context.csv", index=False)
print("saved")

saved


In [9]:
nb = pd.read_csv("data/processed/neighbourhood_summary.csv", dtype={"neighbourhood_id": str})
m = nb.merge(ctx.drop(columns="neighbourhood_name"), on="neighbourhood_id", how="left")
print(m.shape, "| missing:", int(m.isna().sum().sum()))
cols = ["median_s", "mean_dist_km", "median_income", "pct_65_plus", "pct_renters", "pct_apt_5plus", "pct_built_before_1980"]
m[cols].corr(method="spearman").round(2)["median_s"]

(158, 11) | missing: 0


median_s                 1.00
mean_dist_km             0.86
median_income            0.09
pct_65_plus              0.31
pct_renters             -0.36
pct_apt_5plus           -0.03
pct_built_before_1980   -0.12
Name: median_s, dtype: float64

# Stage 5 summary: bringing in the census context

The 2021 census workbook is built for reading, not analysis: one column per neighbourhood, about 2,600 rows of labels, and some labels repeat (the label "65 years and over" appears six times). To avoid pulling the wrong row, I located each number by its section heading plus its row label, and I checked that each heading exists exactly once. I also converted counts to percentages so large and small neighbourhoods can be compared fairly.

The result is one row per neighbourhood with five measures: median household income (2020), share aged 65 and over, share of renters, share living in apartment buildings of five or more storeys, and share of homes built before 1980. There are 158 rows and no missing values, and the census neighbourhood numbers match the boundary file's codes exactly, so the join is clean. I checked one neighbourhood by hand (3,705 renters out of 10,700 households is 34.6%) and it agrees with the workbook. The table is saved as neighbourhood_context.csv.

A first look at how these line up with each neighbourhood's median wait (rank correlation across the 158 neighbourhoods): distance to the nearest station 0.86, income 0.09, share aged 65+ 0.31, renters -0.36, apartments -0.03, and homes built before 1980 -0.12. Income hardly relates to waiting at all, while distance relates strongly. Renters are linked with shorter waits, but renters also live closer to stations (-0.29 correlation with distance), so that link may just reflect dense, central areas and not anything about renting.

A few cautions for later. These are neighbourhood averages, so they say nothing about any single household. The census is a 2021 snapshot applied to incidents from 2011 to 2024. And the measures overlap (income and renters are linked at -0.66, renters and apartments at 0.65), so I'll need to interpret them carefully in the model.
## Some Important Points
**I use percentages and not counts** because a big neighbourhood has more renters than a small one just because it has more people, and a percentage lets neighbourhoods be compared on equal terms.  

**I search by section heading** because the same label appears in several sections, and searching by heading plus label picks the right row and fails loudly if anything changes.